# Session 04 · Aggregation Pipeline

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Setup

In [2]:
from campus_data import seed
from mongo_conn import get_db
db = get_db("campus_agg")
print(seed(db))

def show(rows):
    for r in rows:
        print(r)

{'students': 10, 'batches': 3, 'payments': 12}


## Example 1 - count and average per batch

In [3]:
rows = list(db.students.aggregate([
    {"$group": {"_id": "$batch", "students": {"$sum": 1}, "avg_python": {"$avg": "$marks.python"}}},
    {"$sort": {"_id": 1}},
]))
show({**r, "avg_python": round(r["avg_python"], 1)} for r in rows)
assert [r["students"] for r in rows] == [4, 3, 3]

{'students': 4, 'avg_python': 72.0, '_id': 'AI-01'}
{'students': 3, 'avg_python': 71.7, '_id': 'DS-01'}
{'students': 3, 'avg_python': 77.0, '_id': 'DS-02'}


## Example 2 - computed fields and grades

In [4]:
show(db.students.aggregate([
    {"$addFields": {"total": {"$add": ["$marks.python", "$marks.sql", "$marks.stats"]}}},
    {"$addFields": {"grade": {"$switch": {"branches": [
        {"case": {"$gte": ["$total", 240]}, "then": "A"},
        {"case": {"$gte": ["$total", 180]}, "then": "B"}], "default": "C"}}}},
    {"$project": {"_id": 0, "name": 1, "total": 1, "grade": 1}},
    {"$sort": {"total": -1}},
]))

{'name': 'Faraz Iqbal', 'total': 277, 'grade': 'A'}
{'name': 'Sana Khan', 'total': 265, 'grade': 'A'}
{'name': 'Zoya Hussain', 'total': 249, 'grade': 'A'}
{'name': 'Ayesha Noor', 'total': 248, 'grade': 'A'}
{'name': 'Hina Shah', 'total': 223, 'grade': 'B'}
{'name': 'Ali Raza', 'total': 213, 'grade': 'B'}
{'name': 'Bilal Ahmed', 'total': 197, 'grade': 'B'}
{'name': 'Mariam Javed', 'total': 184, 'grade': 'B'}
{'name': 'Hamza Ali', 'total': 163, 'grade': 'C'}
{'name': 'Usman Tariq', 'total': 137, 'grade': 'C'}


## Example 3 - $unwind: most common skills

In [5]:
top_skills = list(db.students.aggregate([{"$unwind": "$skills"}, {"$group": {"_id": "$skills", "n": {"$sum": 1}}},
                                         {"$sort": {"n": -1, "_id": 1}}, {"$limit": 3}]))
show(top_skills)
assert top_skills[0] == {"_id": "python", "n": 7}

{'n': 7, '_id': 'python'}
{'n': 4, '_id': 'sql'}
{'n': 2, '_id': 'excel'}


## Example 4 - $lookup: fees collected per city

In [6]:
show(db.payments.aggregate([
    {"$lookup": {"from": "students", "localField": "student_id", "foreignField": "student_id", "as": "s"}},
    {"$unwind": "$s"},
    {"$group": {"_id": "$s.city", "collected": {"$sum": "$amount"}}},
    {"$sort": {"collected": -1}},
]))

{'collected': 84000, '_id': 'Karachi'}
{'collected': 48000, '_id': 'Lahore'}
{'collected': 36000, '_id': 'Hyderabad'}
{'collected': 12000, '_id': 'Islamabad'}


## Example 5 - $bucket and $facet

In [7]:
facet = list(db.students.aggregate([{"$facet": {
    "python_bands": [{"$bucket": {"groupBy": "$marks.python", "boundaries": [0, 50, 70, 90, 101],
                                  "output": {"n": {"$sum": 1}}}}],
    "karachi": [{"$match": {"city": "Karachi"}}, {"$count": "n"}],
}}]))[0]
print(facet)

{'python_bands': [{'n': 1, '_id': 0}, {'n': 3, '_id': 50}, {'n': 4, '_id': 70}, {'n': 2, '_id': 90}], 'karachi': [{'n': 5}]}


## Example 6 - top student per batch, saved with $out

In [8]:
db.students.aggregate([
    {"$sort": {"marks.python": -1}},
    {"$group": {"_id": "$batch", "top": {"$first": "$name"}, "score": {"$first": "$marks.python"}}},
    {"$out": "top_students"},
])
show(db.top_students.find().sort("_id", 1))

{'top': 'Zoya Hussain', 'score': 88, '_id': 'AI-01'}
{'top': 'Sana Khan', 'score': 92, '_id': 'DS-01'}
{'top': 'Faraz Iqbal', 'score': 95, '_id': 'DS-02'}
